In [1]:
# %%
import pandas as pd
import numpy as np
import torch
from torch_geometric.data import Data
from sklearn.preprocessing import StandardScaler

print("--- 1. SETUP & WHALE CRUSHER ---")
df = pd.read_csv('HI_Small_1M_Chronological.csv')

# Map Nodes
senders = df['Account'].unique()
receivers = df['Account.1'].unique()
unique_accounts = np.unique(np.concatenate((senders, receivers)))
account_to_id = {account: i for i, account in enumerate(unique_accounts)}
df['Sender_Node_ID'] = df['Account'].map(account_to_id)
df['Receiver_Node_ID'] = df['Account.1'].map(account_to_id)

# Feature Engineering
outgoing = df.groupby('Account').agg(total_sent=('Amount Paid', 'sum'), avg_sent=('Amount Paid', 'mean'), count_sent=('Amount Paid', 'count')).reset_index().rename(columns={'Account': 'Node'})
incoming = df.groupby('Account.1').agg(total_received=('Amount Received', 'sum'), avg_received=('Amount Received', 'mean'), count_received=('Amount Received', 'count')).reset_index().rename(columns={'Account.1': 'Node'})
node_profiles = pd.merge(outgoing, incoming, on='Node', how='outer').fillna(0)

node_profiles['flow_ratio'] = (node_profiles['total_received'] + 1) / (node_profiles['total_sent'] + 1)
for col in ['total_sent', 'avg_sent', 'total_received', 'avg_received']:
    node_profiles[col] = np.log1p(node_profiles[col])

aligned_profiles = pd.merge(pd.DataFrame({'Node': unique_accounts}), node_profiles, on='Node', how='left').fillna(0)
features_array = aligned_profiles[['total_sent', 'avg_sent', 'count_sent', 'total_received', 'avg_received', 'count_received', 'flow_ratio']].values

scaler = StandardScaler()
x_tensor = torch.tensor(scaler.fit_transform(features_array), dtype=torch.float)

# PyG Graph Object
edge_index = torch.tensor(np.array([df['Sender_Node_ID'].values, df['Receiver_Node_ID'].values]), dtype=torch.long)
edge_weights = torch.tensor(np.log1p(df['Amount Paid'].values), dtype=torch.float)
graph_data = Data(x=x_tensor, edge_index=edge_index, edge_attr=edge_weights)

# Parse Typologies ONCE for all evaluations
try:
    df['Timestamp'] = pd.to_datetime(df['Timestamp'], format='mixed')
    df['Account'] = df['Account'].astype(str).str.strip()
    df['Account.1'] = df['Account.1'].astype(str).str.strip()
    
    pattern_data = []
    current_typology = "Unknown"
    with open('HI-Small_Patterns.txt', 'r') as file:
        for line in file:
            line = line.strip()
            if line.startswith("BEGIN LAUNDERING ATTEMPT -"):
                current_typology = line.split('-')[1].split(':')[0].strip()
            elif line.startswith("2022"):
                fields = line.split(',')
                if len(fields) >= 11:
                    pattern_data.append({'Timestamp': fields[0].strip(), 'Account': fields[2].strip(), 'Account.1': fields[4].strip(), 'Mapped_Typology': current_typology})
    
    patterns_df = pd.DataFrame(pattern_data).drop_duplicates()
    patterns_df['Timestamp'] = pd.to_datetime(patterns_df['Timestamp'], format='mixed')
    patterns_df['Account'] = patterns_df['Account'].astype(str).str.strip()
    patterns_df['Account.1'] = patterns_df['Account.1'].astype(str).str.strip()
    
    df = df.merge(patterns_df[['Timestamp', 'Account', 'Account.1', 'Mapped_Typology']], on=['Timestamp', 'Account', 'Account.1'], how='left')
    df['Laundering_Typology'] = df['Mapped_Typology'].fillna("Normal")
    print("Graph Data and Typologies Ready!")
except FileNotFoundError:
    print("Warning: HI-Small_Patterns.txt not found. Typology breakdown will not run.")

--- 1. SETUP & WHALE CRUSHER ---
Graph Data and Typologies Ready!


In [2]:
from torch_geometric.nn import Node2Vec
from sklearn.ensemble import IsolationForest
from sklearn.metrics import f1_score, precision_score, recall_score

print("\n--- 2. BASELINE: PURE STRUCTURAL NODE2VEC ---")
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model_n2v = Node2Vec(graph_data.edge_index, embedding_dim=64, walk_length=10, context_size=5, walks_per_node=5, num_negative_samples=1, p=1.0, q=1.0, sparse=True).to(device)
loader = model_n2v.loader(batch_size=128, shuffle=True, num_workers=0)
optimizer_n2v = torch.optim.SparseAdam(list(model_n2v.parameters()), lr=0.01)

print("Training Node2Vec (Topology only)...")
model_n2v.train()
for epoch in range(1, 11):
    total_loss = 0
    for pos_rw, neg_rw in loader:
        optimizer_n2v.zero_grad()
        loss = model_n2v.loss(pos_rw.to(device), neg_rw.to(device))
        loss.backward()
        optimizer_n2v.step()
        total_loss += loss.item()
    print(f"Epoch {epoch:02d}, Loss: {total_loss / len(loader):.4f}")

model_n2v.eval()
with torch.no_grad():
    node_embeddings = model_n2v(torch.arange(graph_data.num_nodes, device=device)).cpu().numpy()

print("\nRunning Isolation Forest on Node2Vec Embeddings...")
iso_forest_n2v = IsolationForest(n_estimators=100, contamination=0.001, random_state=42)
n2v_predictions = iso_forest_n2v.fit_predict(node_embeddings)
df['Node2Vec_Guess'] = df['Sender_Node_ID'].map(lambda x: np.where(n2v_predictions == -1, 1, 0)[x])

f1 = f1_score(df['Is Laundering'], df['Node2Vec_Guess'], pos_label=1, zero_division=0)
recall = recall_score(df['Is Laundering'], df['Node2Vec_Guess'], pos_label=1, zero_division=0)
print(f"Node2Vec Global F1: {f1:.4f}")
print(f"Criminals Caught: {int(recall * df['Is Laundering'].sum())} out of {df['Is Laundering'].sum()}")


--- 2. BASELINE: PURE STRUCTURAL NODE2VEC ---
Training Node2Vec (Topology only)...
Epoch 01, Loss: 2.9736
Epoch 02, Loss: 1.6747
Epoch 03, Loss: 1.1979
Epoch 04, Loss: 0.9820
Epoch 05, Loss: 0.8797
Epoch 06, Loss: 0.8288
Epoch 07, Loss: 0.8020
Epoch 08, Loss: 0.7863
Epoch 09, Loss: 0.7763
Epoch 10, Loss: 0.7695

Running Isolation Forest on Node2Vec Embeddings...
Node2Vec Global F1: 0.0189
Criminals Caught: 13 out of 575


In [3]:
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
from torch_geometric.utils import negative_sampling
from sklearn.metrics import precision_recall_curve, auc

print("\n--- 3. EXPERIMENT 1: EARLY FUSION (Enriched GraphSAGE) ---")

class EnrichedSAGE(torch.nn.Module):
    def __init__(self):
        super(EnrichedSAGE, self).__init__()
        self.conv1 = SAGEConv(7, 32) # Takes in the 7 financial features!
        self.conv2 = SAGEConv(32, 64)
        
    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        return self.conv2(x, edge_index)

model_early = EnrichedSAGE().to(device)
optimizer_early = torch.optim.Adam(model_early.parameters(), lr=0.01)

print("Training Enriched GraphSAGE (Causes Homophily Trap)...")
for epoch in range(1, 11): 
    model_early.train()
    optimizer_early.zero_grad()
    z = model_early(graph_data.x.to(device), graph_data.edge_index.to(device))
    pos_out = (z[graph_data.edge_index[0].to(device)] * z[graph_data.edge_index[1].to(device)]).sum(dim=-1)
    neg_edge_index = negative_sampling(edge_index=graph_data.edge_index.to(device), num_nodes=z.size(0), num_neg_samples=graph_data.edge_index.size(1))
    neg_out = (z[neg_edge_index[0]] * z[neg_edge_index[1]]).sum(dim=-1)
    loss = F.binary_cross_entropy_with_logits(pos_out, torch.ones_like(pos_out)) + F.binary_cross_entropy_with_logits(neg_out, torch.zeros_like(neg_out))
    loss.backward()
    optimizer_early.step()

model_early.eval()
with torch.no_grad():
    early_embeddings = model_early(graph_data.x.to(device), graph_data.edge_index.to(device)).cpu().numpy()

iso_early = IsolationForest(n_estimators=100, contamination=0.001, random_state=42)
raw_scores_early = -iso_early.fit(early_embeddings).decision_function(early_embeddings)
df['Early_Fusion_Risk'] = df['Sender_Node_ID'].map(lambda x: raw_scores_early[x])

pr_auc_early = auc(*precision_recall_curve(df['Is Laundering'], df['Early_Fusion_Risk'], pos_label=1)[1::-1])
print(f"Early Fusion PR-AUC: {pr_auc_early:.4f}")

for budget in [1000, 5000, 10000]:
    threshold = np.sort(df['Early_Fusion_Risk'])[-budget]
    caught = df[(df['Is Laundering'] == 1) & (df['Early_Fusion_Risk'] >= threshold)].shape[0]
    print(f"Top {budget} Alerts: Caught {caught} criminals.")


--- 3. EXPERIMENT 1: EARLY FUSION (Enriched GraphSAGE) ---
Training Enriched GraphSAGE (Causes Homophily Trap)...
Early Fusion PR-AUC: 0.0009
Top 1000 Alerts: Caught 0 criminals.
Top 5000 Alerts: Caught 52 criminals.
Top 10000 Alerts: Caught 52 criminals.


In [4]:
print("\n--- 4. EXPERIMENT 2: LATE FUSION ---")

# 1. Structural Only Model
structural_x = torch.ones((graph_data.num_nodes, 1), dtype=torch.float).to(device)

class StructuralSAGE(torch.nn.Module):
    def __init__(self):
        super(StructuralSAGE, self).__init__()
        self.conv1 = SAGEConv(1, 32) 
        self.conv2 = SAGEConv(32, 64)
        
    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        return self.conv2(x, edge_index)

model_late = StructuralSAGE().to(device)
optimizer_late = torch.optim.Adam(model_late.parameters(), lr=0.01)

print("Training Structural-Only GraphSAGE...")
for epoch in range(1, 11): 
    model_late.train()
    optimizer_late.zero_grad()
    z = model_late(structural_x, graph_data.edge_index.to(device))
    pos_out = (z[graph_data.edge_index[0].to(device)] * z[graph_data.edge_index[1].to(device)]).sum(dim=-1)
    neg_edge_index = negative_sampling(edge_index=graph_data.edge_index.to(device), num_nodes=z.size(0), num_neg_samples=graph_data.edge_index.size(1))
    neg_out = (z[neg_edge_index[0]] * z[neg_edge_index[1]]).sum(dim=-1)
    loss = F.binary_cross_entropy_with_logits(pos_out, torch.ones_like(pos_out)) + F.binary_cross_entropy_with_logits(neg_out, torch.zeros_like(neg_out))
    loss.backward()
    optimizer_late.step()

model_late.eval()
with torch.no_grad():
    structural_embeddings = model_late(structural_x, graph_data.edge_index.to(device)).cpu().numpy()

# 2. Concat Shape (64D) + Money (7D)
print("Fusing Features and Running Isolation Forest on 71D Profiles...")
late_fusion_embeddings = np.concatenate([structural_embeddings, x_tensor.numpy()], axis=1)

iso_late = IsolationForest(n_estimators=100, contamination=0.001, random_state=42)
raw_scores_late = -iso_late.fit(late_fusion_embeddings).decision_function(late_fusion_embeddings)
df['Late_Fusion_Risk'] = df['Sender_Node_ID'].map(lambda x: raw_scores_late[x])

pr_auc_late = auc(*precision_recall_curve(df['Is Laundering'], df['Late_Fusion_Risk'], pos_label=1)[1::-1])
print(f"Late Fusion PR-AUC: {pr_auc_late:.4f}")

for budget in [1000, 5000, 10000]:
    threshold = np.sort(df['Late_Fusion_Risk'])[-budget]
    caught = df[(df['Is Laundering'] == 1) & (df['Late_Fusion_Risk'] >= threshold)].shape[0]
    print(f"Top {budget} Alerts: Caught {caught} criminals.")


--- 4. EXPERIMENT 2: LATE FUSION ---
Training Structural-Only GraphSAGE...
Fusing Features and Running Isolation Forest on 71D Profiles...
Late Fusion PR-AUC: 0.0009
Top 1000 Alerts: Caught 6 criminals.
Top 5000 Alerts: Caught 11 criminals.
Top 10000 Alerts: Caught 45 criminals.
